In [1]:
!pip install ultralytics==8.0.120

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.7/611.7 kB 14.4 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO

In [11]:
# prompt: dataset is in the /content/parking lot.v5-roboflow-fast-model-augmented3x.yolov11.zip

import zipfile

# Specify the path to your zip file
# Corrected path: removed the double slash and ensured the filename matches the error
zip_file_path = '/content/Parking Lot Availability.v8-roboflow-fast-model-augmented3x.yolov11.zip'

# Specify the directory to extract the contents
dataset_path = '/content/parking_lot_dataset'  # Choose a suitable directory

# Extract the contents of the zip file
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)
    print(f"Dataset extracted to: {dataset_path}")
except FileNotFoundError:
    print(f"Error: The file '{zip_file_path}' was not found. Please ensure it's uploaded to your Colab environment and the path is correct.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")

# Now you can use the extracted dataset with YOLO
# Example using YOLOv8 (assuming you have YOLOv8 installed)
#model = YOLO('yolov8n.pt') # Replace with your model path
#results = model.train(data='/content/parking_lot_dataset/data.yaml', epochs=10) # Replace data.yaml with your config file

Dataset extracted to: /content/parking_lot_dataset


In [4]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

100%|██████████| 6.23M/6.23M [00:00<00:00, 120MB/s]


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL ultralytics.nn.tasks.DetectionModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])` or the `torch.serialization.safe_globals([ultralytics.nn.tasks.DetectionModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of to

In [10]:
import torch
from ultralytics import YOLO

# This context manager tells PyTorch "For these specific lines,
# ignore the 'weights_only' security restriction."
with torch.serialization.safe_globals([torch.nn.modules.container.Sequential]):
    try:
        # We use this trick to bypass the strict check for the legacy version
        model = YOLO('yolov8n.pt')
        print("Model loaded successfully!")
    except Exception as e:
        # If the context manager still trips, we use the environment override
        import os
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
        model = YOLO('yolov8n.pt')
        print("Model loaded via environment override.")

Model loaded successfully!


/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py:518: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(file, map_location='cpu'), file  # load


In [9]:
yolodata_path = dataset_path + "/data.yaml"

In [12]:

# Train the model if you want to fine-tune it on your dataset
model.train(data=dataset_path + "/data.yaml", epochs=10)  # Disable wandb by setting project=None


New https://pypi.org/project/ultralytics/8.4.10 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.0.120 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
yolo/engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/parking_lot_dataset/data.yaml, epochs=10, patience=50, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=None, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=0, resume=False, amp=True, fraction=1.0, profile=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, line_width=None, visualize=False, augment=False, agnostic_nms=False, c

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py:518: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(file, map_location='cpu'), file  # load
/usr/local/lib/python3.12/dist-packages/ultralytics/yolo/utils/checks.py:372: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):
AMP: checks passed ✅
/usr/local/lib/python3.12/dist-packages/ultralytics/yolo/engine/trainer.py:223: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = amp.GradScaler(enabled=self.amp)
train: Scanning /content/parking_lot_dataset/train/labels... 126 images, 0 backgrounds, 0 corrupt: 100%|██

# Run inference on a video file
results = model.predict(source="/content/Đường Phố CHỢ LỚN MÙNG 1 TẾT Ra Sao Từ Hậu Giang Đến Phùng Hưng Lão Tử  Sài Gòn Tết Ất Tỵ 2025 - TheKingFavor (360p, h264, youtube).mp4", save=True, show = True)  # Replace with your video file path